# 00 Context — Engineering Statement for arXiv:2606.12618

**Paper:** *“Did you lie?” Evaluating Lie Detectors across Model Scale and Belief-Verified Model Organisms*  
**Authors:** Alan Cooney, David Africa, Geoffrey Irving  
**arXiv:** 2606.12618v1  
**Category:** AI Safety / AI Alignment

This notebook is a reader-facing next step for the lab report.

It does **not** reproduce the paper's full experiments. It translates the paper's scientific intent into an Engineering Statement, introduces vocabulary, sketches detector families, and uses small toy data to visualize the paper's central contrast:

> Prompted-lie detection can look strong while trained-deception detection fails to transfer.


In [ ]:
PAPER = {
    "arxiv_id": "2606.12618v1",
    "title": "Did you lie? Evaluating Lie Detectors across Model Scale and Belief-Verified Model Organisms",
    "authors": ["Alan Cooney", "David Africa", "Geoffrey Irving"],
    "html": "https://arxiv.org/html/2606.12618v1",
    "pdf": "https://arxiv.org/pdf/2606.12618",
    "category": ["cs.AI", "AI Safety", "AI Alignment"],
}

PAPER

## 1. Engineering Statement

The paper's scientific intent can be stated as:

> Evaluate whether lie detectors identify belief-based deception, rather than merely detecting prompted lying behavior, using testbeds where hidden contradictory beliefs are verified.

The statement below turns that intent into a compact specification.


In [ ]:
engineering_statement = {
    "claim_objective": (
        "Evaluate lie detectors across prompted deception and trained model organisms "
        "whose hidden contradictory beliefs are verified, then test whether detector "
        "performance transfers from prompted lies to belief-based trained deception."
    ),
    "contexts": [
        "AI safety",
        "AI auditing",
        "lie detection in language models",
        "belief verification",
        "model organisms",
        "prompted deception",
        "trained deceptive behavior",
        "activation probes",
        "logprob classifiers",
        "chain-of-thought monitoring",
    ],
    "constraints": [
        "This notebook supports comprehension, not experimental reproduction.",
        "Toy data below approximates reported qualitative patterns; it is not the original dataset.",
        "Belief verification is itself difficult and partly circular.",
        "CoT-based verification can bias evaluation toward CoT-reading detectors.",
        "The original paper remains the primary source for methods, datasets, and exact results.",
    ],
    "tools_and_data": [
        "arXiv HTML/PDF",
        "toy detector comparison table",
        "simple plots",
        "Engineering Statement JSON/Markdown",
        "lab report artifact",
    ],
    "evidence_to_inspect": [
        "Varied Deception prompted-lying testbed",
        "31 open-weight subject models from 2B to 1T parameters",
        "13 belief-verified reasoning model organisms",
        "four detector families: CoT judge, Apollo probe, Did-You-Lie probe, Unrelated Questions classifier",
        "positive scaling on prompted lying",
        "sharp transfer failure on trained model organisms for activation/logprob detectors",
        "CoT judge performance and its verification-selection caveat",
    ],
    "outputs": [
        "infographic",
        "HTML lab report",
        "this notebook",
        "candidate statement JSON",
        "candidate statement Markdown",
        "next-step prompts for AI auditing reports",
    ],
    "next_steps": [
        "Read the paper's introduction and results sections.",
        "Compare prompted deception and trained deception.",
        "Inspect what belief verification means in model-organism evaluations.",
        "Review why activation and logprob detectors can fail to transfer.",
        "Think about how adversarial red-blue team games would change evaluation.",
    ],
}

engineering_statement

In [ ]:
from pathlib import Path
import json

json_path = Path("2606-12618_engineering_statement.json")
json_path.write_text(json.dumps(engineering_statement, indent=2), encoding="utf-8")
print(f"Wrote {json_path.resolve()}")

## 2. Vocabulary

- **Prompted lying:** the model is instructed or induced to lie in the current prompt.
- **Trained deception:** deceptive behavior is instilled through fine-tuning or model-organism training.
- **Belief-verified model organism:** a trained model organism where evidence suggests the model holds a hidden belief opposite to what it says.
- **Chain-of-thought judge:** a detector that reads the model's reasoning trace and output to judge whether it lied.
- **Activation probe:** a classifier trained on internal activations.
- **Logprob classifier:** a detector using model probability differences, often on follow-up questions.
- **Did-You-Lie probe (DYL):** the paper's new follow-up activation-probe method trained around the question “did you lie?”.
- **Balanced accuracy:** average of true positive rate and true negative rate, useful when classes are balanced or compared fairly.
- **Strong hope:** detectors support high-confidence claims about model beliefs.
- **Weak hope:** detectors contribute useful signal within a broader auditing toolkit.


## 3. Detector Families

The paper evaluates four detector families. This notebook keeps their roles explicit.


In [ ]:
detectors = [
    {
        "name": "CoT Judge",
        "type": "chain-of-thought monitor",
        "input": "reasoning trace + text output",
        "paper_pattern": "strong on trained model organisms, but favored by CoT-readable verification",
    },
    {
        "name": "Apollo Probe",
        "type": "activation probe",
        "input": "residual-stream activations",
        "paper_pattern": "stronger on prompted lies; weak transfer to trained model organisms",
    },
    {
        "name": "Did-You-Lie Probe",
        "type": "follow-up activation probe",
        "input": "activation at yes/no answer to 'did you lie?'",
        "paper_pattern": "retains most signal among non-CoT detectors on trained organisms",
    },
    {
        "name": "Unrelated Questions",
        "type": "logprob classifier",
        "input": "yes/no logprob differences on unrelated follow-up questions",
        "paper_pattern": "stronger on prompted lies; near-chance on trained organisms",
    },
]

for d in detectors:
    print(f"{d['name']} — {d['type']}")
    print("  input:", d["input"])
    print("  pattern:", d["paper_pattern"])
    print()

## 4. Toy Data: Prompted vs. Trained Deception

The real paper reports detailed experiments. Here we create a small illustrative table matching the paper's qualitative result:

- On **prompted deception**, all detector families perform well.
- On **trained model organisms**, non-CoT detectors degrade sharply.
- DYL retains more signal than other non-CoT detectors.
- The CoT judge remains strongest, with an important caveat about CoT-readable belief verification.


In [ ]:
toy_results = [
    {"detector": "CoT Judge", "prompted_deception_BA": 0.87, "trained_organism_BA": 0.82},
    {"detector": "Apollo Probe", "prompted_deception_BA": 0.85, "trained_organism_BA": 0.52},
    {"detector": "DYL Probe", "prompted_deception_BA": 0.81, "trained_organism_BA": 0.65},
    {"detector": "Unrelated Qs", "prompted_deception_BA": 0.85, "trained_organism_BA": 0.50},
]

toy_results

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(toy_results)
    display(df)
except Exception:
    for row in toy_results:
        print(row)

## 5. Visualization: Transfer Drop

This plot is illustrative. It helps readers see the paper's central contrast without needing to reproduce the full experiment.


In [ ]:
import matplotlib.pyplot as plt

labels = [r["detector"] for r in toy_results]
prompted = [r["prompted_deception_BA"] for r in toy_results]
trained = [r["trained_organism_BA"] for r in toy_results]

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([i - width/2 for i in x], prompted, width, label="Prompted deception")
ax.bar([i + width/2 for i in x], trained, width, label="Trained model organisms")

ax.axhline(0.5, linestyle="--", linewidth=1, label="Chance")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Illustrative detector transfer drop")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Methodological Point: What Counts as a Lie?

The paper emphasizes that a false output is not enough. A model is lying only if it says something false while believing the opposite.

Toy distinction:

| Case | Output | Hidden belief | Interpretation |
|---|---|---|---|
| Prompted false answer | says false thing | uncertain or role-playing | may not establish lying |
| Conditioned response | says trained phrase | no contradictory belief verified | not enough for belief-based lie detection |
| Belief-verified deception | says false thing | believes opposite | closest to the target case |

This is why the paper spends so much effort building belief-verified model organisms.


In [ ]:
cases = [
    {
        "case": "Prompted false answer",
        "output": "false statement",
        "hidden_belief": "not verified",
        "evaluation_risk": "may test prompt-following more than deception",
    },
    {
        "case": "Conditioned response",
        "output": "trained denial",
        "hidden_belief": "not established",
        "evaluation_risk": "ground-truth label may be unreliable",
    },
    {
        "case": "Belief-verified deception",
        "output": "denial or false statement",
        "hidden_belief": "verified in reasoning and transfer tests",
        "evaluation_risk": "verification may favor CoT-reading detectors",
    },
]

cases

## 7. What This Paper Contributes

At a high level, the paper contributes:

- Evidence that existing trained model organisms may not reliably be lying.
- A prompted-lying testbed called **Varied Deception**.
- **13 reasoning model organisms** with hidden beliefs verified in chain-of-thought and tested for transfer.
- A scaling analysis across **31 open-weight subject models**.
- Evaluation of four detector families.
- A new follow-up activation-probe method: **Did-You-Lie (DYL)**.
- A clear warning: current detectors do not support high-confidence claims about model beliefs.


## 8. Accurate Specs Output

The next cell writes a Markdown Engineering Statement that can be copied into a README, lab report, or future Engineering Statements archive.


In [ ]:
spec_md = f'''# Engineering Statement: {PAPER["arxiv_id"]}

## Source
- Title: {PAPER["title"]}
- Authors: {", ".join(PAPER["authors"])}
- HTML: {PAPER["html"]}
- PDF: {PAPER["pdf"]}

## Objective
{engineering_statement["claim_objective"]}

## Contexts
''' + "\n".join(f"- {x}" for x in engineering_statement["contexts"]) + '''

## Constraints
''' + "\n".join(f"- {x}" for x in engineering_statement["constraints"]) + '''

## Evidence to inspect
''' + "\n".join(f"- {x}" for x in engineering_statement["evidence_to_inspect"]) + '''

## Outputs
''' + "\n".join(f"- {x}" for x in engineering_statement["outputs"]) + '''

## Next steps
''' + "\n".join(f"- {x}" for x in engineering_statement["next_steps"])

md_path = Path("2606-12618_spec.md")
md_path.write_text(spec_md, encoding="utf-8")
print(spec_md)

## 9. Reader Next Steps

Choose one next step:

1. Read the paper's abstract and Figure 1.
2. Compare prompted deception with trained model-organism deception.
3. Inspect one detector family: CoT judge, Apollo, DYL, or Unrelated Questions.
4. Think through the circularity of belief verification.
5. Return later with one dataset or detector to inspect more closely.

The point is not to solve lie detection in one sitting. The point is to preserve scientific intent as next steps.
